<a href="https://colab.research.google.com/github/ubaid8878/Flyrank-ML-Internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ubaid8878/Flyrank-ML-Internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [ ]:
## My ranked action playbook

The baseline ranks pages using observable signals available before an action is taken. The purpose is to prioritize pages for human review, not to automatically change content.

Reason codes:
- STALE_LOW_CTR: page is old and has low CTR, so review refresh and CTR opportunities.
- STALE: page is old but does not have low CTR.
- LOW_CTR: page has low CTR but is not stale.
- NO_FLAG: no baseline action is recommended.

The highest-ranked pages receive the strongest combination of observable signals. These rankings are decision-support rather than proof that a page will improve after an intervention.

In [2]:
import os
import subprocess

REPO_DIR = "/content/Flyrank-ML-Internship"
REPO_URL = "https://github.com/ubaid8878/Flyrank-ML-Internship"

# Clone your repository if it is not already in Colab
if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", REPO_URL, REPO_DIR],
        check=True
    )

# Move into your repository
os.chdir(REPO_DIR)

print("Current folder:", os.getcwd())
print("Dataset exists:", os.path.exists(
    "data/raw/content_refresh_anonymized.csv"
))

Current folder: /content/Flyrank-ML-Internship
Dataset exists: True


In [3]:
import os
import pandas as pd
import numpy as np

# Load the starter dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Median CTR used as the simple baseline threshold
ctr_median = df["ctr"].median()

print("CTR median:", round(ctr_median, 4))

# Observable conditions
stale = df["days_since_last_update"] >= 180
low_ctr = df["ctr"] < ctr_median

# One reason code per page
df["reason_code"] = np.select(
    [
        stale & low_ctr,
        stale,
        low_ctr
    ],
    [
        "STALE_LOW_CTR",
        "STALE",
        "LOW_CTR"
    ],
    default="NO_FLAG"
)

# Score: stronger combined signal gets higher priority
df["baseline_score"] = (
    stale.astype(int) * 2 +
    low_ctr.astype(int)
)

# Action mapping
df["action"] = np.select(
    [
        df["reason_code"] == "STALE_LOW_CTR",
        df["reason_code"] == "STALE",
        df["reason_code"] == "LOW_CTR"
    ],
    [
        "REVIEW_REFRESH_AND_CTR",
        "REVIEW_REFRESH",
        "REVIEW_CTR"
    ],
    default="NO_ACTION"
)

# Rank
df["rank"] = (
    df["baseline_score"]
    .rank(method="first", ascending=False)
    .astype(int)
)

# Ranked queue
queue = df.sort_values(
    ["baseline_score", "impressions_90d"],
    ascending=[False, False]
).copy()

print("\nReason code counts:")
print(queue["reason_code"].value_counts())

print("\nTop 20:")
print(
    queue[
        [
            "content_id",
            "baseline_score",
            "reason_code",
            "action",
            "days_since_last_update",
            "ctr",
            "impressions_90d"
        ]
    ].head(20).to_string(index=False)
)

CTR median: 0.07

Reason code counts:
reason_code
NO_FLAG          15136
LOW_CTR          14690
STALE_LOW_CTR      120
STALE               54
Name: count, dtype: int64

Top 20:
          content_id  baseline_score   reason_code                 action  days_since_last_update  ctr  impressions_90d
content_5feee3994adb               3 STALE_LOW_CTR REVIEW_REFRESH_AND_CTR                     194 0.01             7812
content_b16bd7307b39               3 STALE_LOW_CTR REVIEW_REFRESH_AND_CTR                     194 0.00             4590
content_074ba6ead17b               3 STALE_LOW_CTR REVIEW_REFRESH_AND_CTR                     183 0.00              533
content_fd16e3475c29               3 STALE_LOW_CTR REVIEW_REFRESH_AND_CTR                     183 0.00              429
content_6476d1d8c050               3 STALE_LOW_CTR REVIEW_REFRESH_AND_CTR                     313 0.00              304
content_4f241bad48a3               3 STALE_LOW_CTR REVIEW_REFRESH_AND_CTR                     236 0.00 

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [ ]:
## Intended use and limits

### Intended use

This playbook is intended to help an SEO/content team prioritize pages for human review. It provides a ranked queue based on observable signals such as staleness, CTR, impressions, position, and content characteristics.

The ranking can help a reviewer decide which pages deserve attention first.

### Limits

The score does not predict guaranteed traffic gains. It does not prove that a page is declining because it is stale or because of CTR. The relationships observed in this dataset are directional.

The baseline has not been tested as a production decision system. Results may vary across clients, topics, search environments, and time periods.

The score should therefore be treated as decision-support rather than an automatic content optimization system.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [ ]:
## Human review rules and no-go list

Every ranked recommendation should be reviewed by a person before an action is taken.

### Human review checklist

A reviewer should check:

1. Whether the page is still relevant to the intended search need.
2. Whether the content is factually accurate and current.
3. Whether the page already performs well for an important query.
4. Whether the CTR signal is based on enough impressions to be meaningful.
5. Whether a refresh could remove useful information or change the page's search intent.
6. Whether the recommended action is appropriate for the business and audience.

### What should NOT be automated

The system should not automatically:

- rewrite or delete content;
- change search intent;
- publish SEO changes without review;
- redirect or remove pages;
- make claims about Google's ranking algorithm;
- treat low CTR as proof that a page is bad;
- guarantee traffic or ranking improvements.

The score should only prioritize work for human review.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [ ]:
## Monitoring and retrain triggers

The playbook should be monitored rather than assumed to remain valid forever.

Useful triggers include:

- Precision@20 or Precision@50 falls materially compared with the validated benchmark.
- The distribution of CTR, impressions, page age, or other important features changes substantially.
- The proportion of pages receiving each reason code changes unexpectedly.
- New content types or search behaviors make the existing thresholds less representative.
- Human reviewers repeatedly reject the same type of recommendation.
- A new period of labeled data becomes available and provides enough examples for re-evaluation.

A retraining or threshold review should be considered when these signals indicate that the existing ranking no longer represents current data well. Monitoring is intended as a practical safeguard, not a guarantee of future performance.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [4]:
# Create output directory
os.makedirs("work/outputs", exist_ok=True)

# Keep the paper-facing queue focused on useful columns
paper_queue = queue[
    [
        "rank",
        "content_id",
        "baseline_score",
        "reason_code",
        "action",
        "days_since_last_update",
        "ctr",
        "impressions_90d"
    ]
].copy()

# Export
output_path = "work/outputs/baseline_action_score.csv"

paper_queue.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows exported:", len(paper_queue))

print("\nTop 10 exported actions:")
print(paper_queue.head(10).to_string(index=False))

Saved: work/outputs/baseline_action_score.csv
Rows exported: 30000

Top 10 exported actions:
 rank           content_id  baseline_score   reason_code                 action  days_since_last_update  ctr  impressions_90d
   46 content_5feee3994adb               3 STALE_LOW_CTR REVIEW_REFRESH_AND_CTR                     194 0.01             7812
    2 content_b16bd7307b39               3 STALE_LOW_CTR REVIEW_REFRESH_AND_CTR                     194 0.00             4590
   14 content_074ba6ead17b               3 STALE_LOW_CTR REVIEW_REFRESH_AND_CTR                     183 0.00              533
   16 content_fd16e3475c29               3 STALE_LOW_CTR REVIEW_REFRESH_AND_CTR                     183 0.00              429
   59 content_6476d1d8c050               3 STALE_LOW_CTR REVIEW_REFRESH_AND_CTR                     313 0.00              304
    5 content_4f241bad48a3               3 STALE_LOW_CTR REVIEW_REFRESH_AND_CTR                     236 0.00              285
  113 content_ea41fe5cf29

In [ ]:
## Cost/value thinking

The ranked queue should be used to spend limited editorial time where the potential value of review is highest.

Pages with strong visibility and a combined stale + low-CTR signal may deserve earlier review because changes to visible pages could have greater potential value. However, impressions are only a prioritization signal and do not guarantee business value.

A practical workflow is to review a small number of high-ranked pages first, record whether the recommendation was useful, and then expand the workflow only if the results justify the additional editorial cost.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.